In [1]:
import json
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import inspect
activations = json.load(open("../activation_sample.json"))
model_id = "google/gemma-3-27b-it"

In [2]:
feats = torch.tensor(activations["Layer 10, Prompt: [[{\"role\": \"system\", \"content\": [{\"type\": \"text\", \"text\": \"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of a female person, whether that manifests in direct words, nouns, pronouns, etc\"}]}, {\"role\": \"user\", \"content\": [{\"type\": \"text\", \"text\": \"Elena adjusted her glasses and peered at the complex architectural blueprints. She had spent months drafting these designs for the new museum. By the time the sun set, the architect felt a profound sense of accomplishment.\"}]}], [{\"role\": \"system\", \"content\": [{\"type\": \"text\", \"text\": \"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of a female person, whether that manifests in direct words, nouns, pronouns, etc\"}]}, {\"role\": \"user\", \"content\": [{\"type\": \"text\", \"text\": \"A young girl stood by the edge of the rushing river, her hair whipping in the wind. She watched the salmon leap against the current with wide-eyed wonder. Nature always seemed to offer her a peculiar form of solace.\"}]}]]"])

In [93]:
feats.shape

torch.Size([175, 86016])

In [3]:
processor = AutoProcessor.from_pretrained(model_id)

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [4]:
messages = eval(list(activations.keys())[0].split("Layer 10, Prompt: ")[1])

In [5]:
messages

[[{'role': 'system',
   'content': [{'type': 'text',
     'text': 'You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of a female person, whether that manifests in direct words, nouns, pronouns, etc'}]},
  {'role': 'user',
   'content': [{'type': 'text',
     'text': 'Elena adjusted her glasses and peered at the complex architectural blueprints. She had spent months drafting these designs for the new museum. By the time the sun set, the architect felt a profound sense of accomplishment.'}]}],
 [{'role': 'system',
   'content': [{'type': 'text',
     'text': 'You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of a female person, whether that manifests in direct words, nouns, pronouns, etc'}]},
  {'role': 'user',
   'content': [{'type': 'text',
     'text': 'A young girl stood by the edge of the rushing river, her hair whipping in the wind. She watched the salmon leap against the c

In [6]:
classified = {"0": {"labels": {"Elena": "1", " adjusted": "0", " her": "1", " glasses": "0", " and": "0", " peered": "0", " at": "0", " the": "0", " complex": "0", " architectural": "0", " blueprints": "0", ".": "0", " She": "1", " had": "0", " spent": "0", " months": "0", " drafting": "0", " these": "0", " designs": "0", " for": "0", " new": "0", " museum": "0", " By": "0", " time": "0", " sun": "0", " set": "0", ",": "0", " architect": "0", " felt": "0", " a": "0", " profound": "0", " sense": "0", " of": "0", " accomplishment": "0"}}, "1": {"labels": {"A": "0", " young": "0", " girl": "1", " stood": "0", " by": "0", " the": "0", " edge": "0", " of": "0", " rushing": "0", " river": "0", ",": "0", " her": "1", " hair": "0", " whipping": "0", " in": "0", " wind": "0", ".": "0", " She": "1", " watched": "0", " salmon": "0", " leap": "0", " against": "0", " current": "0", " with": "0", " wide": "0", " -": "0", "eyed": "0", " wonder": "0", " Nature": "0", " always": "0", " seemed": "0", " to": "0", " offer": "0", " a": "0", " peculiar": "0", " form": "0", " solace": "0"}}}

In [89]:
inputs = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            )
tokenized = [
    [processor.decode(id) for id in inputs["input_ids"][b]]
    for b in range(len(inputs["input_ids"]))
]

concept_token_indices = [
                j+len(tokenized[0])*k 
                for k in range(len(tokenized))
                for j, token in enumerate(tokenized[k])
                if token in classified[f"{k}"]["labels"]
                and classified[f"{k}"]["labels"][token] == "1"
            ]

In [92]:
inputs["input_ids"].shape

torch.Size([2, 91])

In [94]:
2*91

182

In [83]:
classified

{'0': {'labels': {'Elena': '1',
   ' adjusted': '0',
   ' her': '1',
   ' glasses': '0',
   ' and': '0',
   ' peered': '0',
   ' at': '0',
   ' the': '0',
   ' complex': '0',
   ' architectural': '0',
   ' blueprints': '0',
   '.': '0',
   ' She': '1',
   ' had': '0',
   ' spent': '0',
   ' months': '0',
   ' drafting': '0',
   ' these': '0',
   ' designs': '0',
   ' for': '0',
   ' new': '0',
   ' museum': '0',
   ' By': '0',
   ' time': '0',
   ' sun': '0',
   ' set': '0',
   ',': '0',
   ' architect': '0',
   ' felt': '0',
   ' a': '0',
   ' profound': '0',
   ' sense': '0',
   ' of': '0',
   ' accomplishment': '0'}},
 '1': {'labels': {'A': '0',
   ' young': '0',
   ' girl': '1',
   ' stood': '0',
   ' by': '0',
   ' the': '0',
   ' edge': '0',
   ' of': '0',
   ' rushing': '0',
   ' river': '0',
   ',': '0',
   ' her': '1',
   ' hair': '0',
   ' whipping': '0',
   ' in': '0',
   ' wind': '0',
   '.': '0',
   ' She': '1',
   ' watched': '0',
   ' salmon': '0',
   ' leap': '0',
   ' 

In [90]:
concept_token_indices

[46, 48, 58, 134, 144, 151, 170]

In [78]:
classified["0"]["labels"][tokenized[0][47]]

'0'

In [60]:
for k in range(len(tokenized)):
    for j, token in enumerate(tokenized[k]):
        # print(j,token)
        print(k)

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


In [8]:
all_feats = feats.clone()
feats = feats.flatten(end_dim=1)[concept_token_indices]

top_feature_values, top_feature_indices = (
                        ((feats.abs().sum(dim=0)+feats.abs().flatten().mean()*(feats>1e-3).sum(dim=0))/(all_feats.flatten(end_dim=1).abs().sum(dim=0)+1e-5)).topk(20)
                    )

In [19]:
vis = feats[:, [16107, 82495, 56304, 51497, 83679, 52910, 32561, 28231, 22098, 64325, 55812, 34287, 25760, 12065, 68537, 17061,  2840, 36487, 11109,  3583]].detach().cpu().permute((1,0)).numpy()

In [53]:
concept_token_indices

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 47,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 54,
 55,
 56,
 57,
 58,
 59,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90]

In [52]:
inputs["input_ids"][0].shape

torch.Size([91])

In [40]:
tokenized_one = tokenized[0].copy()

tokenized_one.extend(tokenized[1].copy())

padding_removed = tokenized_one[5:]

In [48]:
padding_removed

['<bos>',
 '<start_of_turn>',
 'user',
 '\n',
 'You',
 ' are',
 ' a',
 ' linguistics',
 ' expert',
 ',',
 ' your',
 ' task',
 ' is',
 ' to',
 ' identify',
 ' all',
 ' words',
 ' that',
 ' fall',
 ' under',
 ' the',
 ' linguistic',
 ' umbrella',
 ' of',
 ' a',
 ' female',
 ' person',
 ',',
 ' whether',
 ' that',
 ' manifests',
 ' in',
 ' direct',
 ' words',
 ',',
 ' nouns',
 ',',
 ' pronouns',
 ',',
 ' etc',
 '\n\n',
 'Elena',
 ' adjusted',
 ' her',
 ' glasses',
 ' and',
 ' peered',
 ' at',
 ' the',
 ' complex',
 ' architectural',
 ' blueprints',
 '.',
 ' She',
 ' had',
 ' spent',
 ' months',
 ' drafting',
 ' these',
 ' designs',
 ' for',
 ' the',
 ' new',
 ' museum',
 '.',
 ' By',
 ' the',
 ' time',
 ' the',
 ' sun',
 ' set',
 ',',
 ' the',
 ' architect',
 ' felt',
 ' a',
 ' profound',
 ' sense',
 ' of',
 ' accomplishment',
 '.',
 '<end_of_turn>',
 '\n',
 '<start_of_turn>',
 'model',
 '\n',
 '<bos>',
 '<start_of_turn>',
 'user',
 '\n',
 'You',
 ' are',
 ' a',
 ' linguistics',
 ' expert

In [51]:
feats.shape

torch.Size([175, 86016])